In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
cd ..

/Users/leonmac/Codes/BO-Set


# Synthetic Experiments
- BA graph ws grid: 100 nodes
- all the kernels
- Plot r-1
- Plot validation error plot

In [3]:
import numpy as np
import torch
import random
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import networkx as nx
import itertools
import multiprocessing
import yaml
import time
import os
from math import comb
from functools import partial
from scipy.stats import spearmanr
from tqdm.std import tqdm

import utils.config_utils as config_utils
from problems.underlying_problem import get_synthetic_problem
from search.trust_region import restart
from search.utils import eigendecompose_laplacian
from search.models import initialize_model
from botorch.utils.transforms import standardize

np.random.seed(0)
random.seed(0)
seed=0

In [4]:
### Define graph
def underlying_graph(config):
    # graph_type = grid, ba, ws
    base_problem = get_synthetic_problem(
        label=config["problem_name"], 
        seed=0,
        problem_kwargs=config["problem_settings"])
    return base_problem

def initialize(Graph):
    X_train = torch.empty(0,1)
    Y_train = torch.empty(0,1)
    base_model, base_mll, based_cached_eigenbasis = initialize_model(
                    train_X=X_train,
                    train_Y=Y_train,
                    context_graph=Graph,
                    covar_type="diffusion",
                    covar_kwargs = {"order": 1},
                    fit_model=False,
                    ard=True,
                    use_fixed_noise=False,
                    use_saas_map=False)
    pos = nx.kamada_kawai_layout(Graph)
    return base_model, pos # base_mll, based_cached_eigenbasis

def output_model(covar_type, X_train, Y_train, Graph):
    if covar_type in ["polynomial", "polynomial_suminverse"]:
        order = None
        model, mll, cached_eigenbasis = initialize_model(
                    train_X=X_train,
                    train_Y=Y_train,
                    context_graph=Graph,
                    covar_type=covar_type,
                    covar_kwargs = {"order": order},
                    fit_model=True,
                    ard=True,
                    use_fixed_noise=False,
                    optim_kwargs = {"train_iters": 1500,
                                    "lr": .1},
                    use_saas_map=False)
    elif covar_type == "diffusion":
        order = None
        model, mll, cached_eigenbasis = initialize_model(
                    train_X=X_train,
                    train_Y=Y_train,
                    context_graph=Graph,
                    covar_type=covar_type,
                    covar_kwargs = {"order": order},
                    fit_model=True,
                    ard=False,
                    use_fixed_noise=False,
                    optim_kwargs = {"train_iters": 1500,
                                    "lr": .1},
                    use_saas_map=False)
    elif covar_type == "diffusion_ard":
        order = Graph.number_of_nodes()
        model, mll, cached_eigenbasis = initialize_model(
                    train_X=X_train,
                    train_Y=Y_train,
                    context_graph=Graph,
                    covar_type="diffusion",
                    covar_kwargs = {"order": None},
                    fit_model=True,
                    ard=True,
                    use_fixed_noise=False,
                    optim_kwargs = {"train_iters": 1500,
                                    "lr": .1},
                    use_saas_map=False)
    return model, mll, cached_eigenbasis

In [5]:
# Find the neighbours of a given ComboNode derived from the underlying Graph
def NeighbourFinder(Graph, ComboNode): # Output edges connected to this ComboNode
    neighbours = np.empty(shape=(0,len(ComboNode)),dtype=int) # Initialise an empty neighbour np.array
    for idx, i in enumerate(ComboNode):
        ComboNode_list = np.delete(ComboNode,idx) # Get the elements except i
        # Find node i's neighbours in the original graph excluding nodes from ComboNode_list
        i_original_neighbours = np.array([j for j in list(Graph.neighbors(i)) if j not in ComboNode_list])
        # Construct the combinatorial nodes for node i
        i_combo_neighbours = np.array(list(map(lambda x : np.append(ComboNode_list, x), i_original_neighbours)))
        if len(i_combo_neighbours)!=0: # stack the neighbours if it is not none
            i_combo_neighbours.sort(axis=1) # sort the array to create a unique identifier for each combo node
            neighbours = np.vstack((neighbours, i_combo_neighbours))
    # Construct the edges from this ComboNode to its neighbours
    stack = np.repeat(np.array(ComboNode).reshape(1,-1),len(neighbours),axis=0)
    edge_array = np.stack((neighbours, stack),axis=1)
    edges = list(tuple(map(tuple, i)) for i in edge_array) # Convert np.array to list of tuples
    return edges

# Find the ComboSubgraph of size Q centred at a given ComboNode by gradually including multi-hops of neighbours
# Initialise the Combo_Subgraph as nx.Graph() and ComboNode_list as [ComboNode] at beginning 
def ComboSubgraph_Constructor(Graph, ComboSubgraph, ComboNode_list, Q=200, l=1): 
    previous_ComboSubgraph_set = set(ComboSubgraph.nodes())
    for ComboNode in ComboNode_list: # add new neighbours of the previous neighbours to the previous ComboSubgraph
        ComboSubgraph.add_edges_from(NeighbourFinder(Graph, ComboNode))
    new_neighbours = list(set(ComboSubgraph.nodes()) - previous_ComboSubgraph_set)
    print(f"new neighbours at hop {l}: {new_neighbours}")
    if ComboSubgraph.number_of_nodes()<=Q and len(new_neighbours)>0: # do a recursive operation if ComboSubgraph's size < Q
        # Here we set all the neighbours at the current hop as the ComboNode_list for next step
        return ComboSubgraph_Constructor(Graph, ComboSubgraph, new_neighbours, Q, l+1)
    else: # randomly remove a subset of neighbours at the current hop to reach Q 
        nodes_to_remove = random.sample(new_neighbours, 
                                        k=ComboSubgraph.number_of_nodes()-Q)
        ComboSubgraph.remove_nodes_from(nodes_to_remove)
        return ComboSubgraph

# Map the indices of nodes on combinatorial graph to numerical indices
def Index_Mapper(Graph):  # e.g. (2,8,13)-> 3, we need numerical indices for kernel computation
    node_list = [i for i in Graph.nodes()]
    mapping = {old_node: new_idx for new_idx, old_node in enumerate(node_list, start=0)}
    return nx.relabel_nodes(Graph, mapping)

# Construct the entire ComboGraph for k-node combinations given an underlying Graph.
def ComboGraph_Constructor(Graph,k,node=None):
    ComboGraph = nx.Graph() # Initialise the ComboGraph
    total_comb = comb(Graph.number_of_nodes(), k)
    print("Number of ComboNodes:", total_comb)
    if node == None:
        nodes_selfcombo = list(itertools.combinations(list(Graph.nodes()),k)) # All the possible ComboNodes
        # Use multiprocessing to parallel compute each ComboNode's edges
        partial_function = partial(NeighbourFinder, Graph)
        with multiprocessing.Pool(8) as pool:
            ComboEdges = pool.map(partial_function, nodes_selfcombo)
    for x in ComboEdges:
        ComboGraph.add_edges_from(x)
    print("Number of ComboEdges:", ComboGraph.number_of_edges())
    return ComboGraph, total_comb

In [6]:
# Define signal
def underlying_signal(base_model, j=2):
    all_X = torch.arange(len(base_problem.underlying_graph)).to(torch.float)
    all_Y = base_model.covar_module.base_kernel.eigenbasis[:,j].detach().clone()
    return all_X, all_Y

def get_signal(ComboGraph, original_Y, prop_data = 0.5):
    all_Y = torch.tensor([original_Y[list(i)].mean() for i in ComboGraph.nodes()])
    ComboSubGraph_mapped = Index_Mapper(ComboSubgraph)
    all_X = torch.arange(len(ComboSubGraph_mapped)).to(torch.float)
    
    from botorch.utils.transforms import standardize
    n_init = len(all_X)
    X = all_X.reshape(-1, 1).detach().clone()

    # Y = base_problem(X.reshape(-1, 1))
    Y = all_Y.reshape(-1, 1).to(dtype=torch.float).detach().clone()
    Y = standardize(Y)

    best_loc = Y.argmax().item()
    X_best = X[best_loc]

    n_train = int(n_init * prop_data)
    ntrain_indices = np.random.choice(n_init, n_train, replace=False)
    ntest_indices = np.array([i for i in range(n_init) if i not in ntrain_indices])
    X_train, Y_train = X[ntrain_indices, ...], Y[ntrain_indices]
    X_test, Y_test = X[ntest_indices, ...], Y[ntest_indices, ...]
    return all_X, Y.squeeze(), X_train, Y_train, X_test, Y_test, X_best

def get_signal_noise(ComboGraph, original_Y, prop_data = 0.5, sigma = 0.1):
    all_Y = torch.tensor([original_Y[list(i)].mean() for i in ComboGraph.nodes()])
    ComboSubGraph_mapped = Index_Mapper(ComboSubgraph)
    all_X = torch.arange(len(ComboSubGraph_mapped)).to(torch.float)

    n_init = len(all_X)
    X = all_X.reshape(-1, 1).detach().clone()

    # Y = base_problem(X.reshape(-1, 1))
    Y = all_Y.reshape(-1, 1).to(dtype=torch.float).detach().clone()
    Y = standardize(Y)
    Y += torch.tensor(np.random.normal(scale=sigma, size = Y.shape))

    best_loc = Y.argmax().item()
    X_best = X[best_loc]

    n_train = int(n_init * prop_data)
    ntrain_indices = np.random.choice(n_init, n_train, replace=False)
    ntest_indices = np.array([i for i in range(n_init) if i not in ntrain_indices])
    X_train, Y_train = X[ntrain_indices, ...], Y[ntrain_indices]
    X_test, Y_test = X[ntest_indices, ...], Y[ntest_indices, ...]
    return all_X, Y.squeeze(), X_train, Y_train, X_test, Y_test, X_best

In [7]:
def plot_signal(ax_signal, Graph, signal, pos, y_label="(a)"):
    nodes = Graph.nodes()
    node_size=50
    nx.draw_networkx(Graph, pos, with_labels=False, ax = ax_signal, node_size=node_size)
    nc = nx.draw_networkx_nodes(Graph, pos, nodelist=nodes, node_color=signal, node_size=node_size, cmap=mpl.colormaps['Blues'], ax = ax_signal)
    #plt.colorbar(nc, ax=ax_signal, boundaries=np.linspace(-0.25,0.25,11))
    plt.colorbar(nc, ax=ax_signal)
    ax_signal.set_axis_off()
    ax_signal.text(-0.9, 0.8, y_label, fontsize=20.)

def plot_validation(ax_kernel, model, X_test, Y_test, title = None):
    model.eval()
    with torch.no_grad():
        # predict on train input
        Y_test_pred = model.posterior(X_test)
        Y_test_pred_mean, Y_test_pred_std = Y_test_pred.mean.flatten(), torch.sqrt(Y_test_pred.variance).flatten()
        ax_kernel.plot(Y_test, Y_test_pred_mean, ".")
        ax_kernel.errorbar(Y_test, Y_test_pred_mean, yerr=Y_test_pred_std, linestyle="None", color="C0", alpha=0.3)
        ax_kernel.plot([Y_test.min(), Y_test.max()], [Y_test.min(), Y_test.max()], color="gray", linestyle='--')
        ax_kernel.text(-2, 2, r'$\rho$' + f" : {round(spearmanr(Y_test_pred_mean, Y_test)[0], 2)}")
        ax_kernel.set_xlabel("Validation ground truth")
        if title is not None:
            ax_kernel.set_title(title)

def plot_rinverse(ax_kernel, model):
    ax_kernel.plot(model.covar_module.base_kernel.eigenvalues.detach(), torch.diag(model.covar_module.base_kernel.get_dist()).detach())
    ax_kernel.ticklabel_format(style='sci', scilimits=(-1,1))

In [8]:
# Get the underlying function f: the Eigenvector that correspondes to the j-th Eigenvalue
yaml_path = 'configurations/testing_synthetic.yaml'
config = config_utils.setup(yaml_path)
j=2

base_problem = underlying_graph(config)
base_model, pos = initialize(base_problem.underlying_graph)
original_X, original_Y = underlying_signal(base_model, j=j)

Using BA network (n=100 m=2) with eigenvector_centrality
feature_computation...........


/Users/leonmac/anaconda3/lib/python3.11/site-packages/botorch/models/gp_regression.py:161: UserWarning: The model inputs are of type torch.float32. It is strongly recommended to use double precision in BoTorch, as this improves both precision and stability and can help avoid numerical errors. See https://github.com/pytorch/botorch/discussions/1444
  self._validate_tensor_args(X=transformed_X, Y=train_Y, Yvar=train_Yvar)


In [9]:
# Get the synthetic problem: the mean of f across sets of k-node combinations on a random ComboSubgraph of size Q
random.seed(0)
k, Q = 3, 100
ComboNode = random.sample(list(base_problem.underlying_graph.nodes), k) # Sample a random ComboNode as ComboSubgraph centre
ComboNode.sort() # Very important to sort the ComboNode!
ComboNode = tuple(ComboNode)
print("Number of all possible ComboNodes:", comb(base_problem.underlying_graph.number_of_nodes(), k))
print("ComboNode for the ComboSubgraph Centre:", ComboNode)
ComboSubgraph = ComboSubgraph_Constructor(base_problem.underlying_graph, nx.Graph(), [ComboNode], Q=Q)
_, pos_ = initialize(ComboSubgraph)

all_X, all_Y, X_train, Y_train, X_test, Y_test, X_best = get_signal(ComboSubgraph, original_Y, prop_data = 0.5)

Number of all possible ComboNodes: 161700
ComboNode for the ComboSubgraph Centre: (49, 53, 97)
new neighbours at hop 1: [(14, 49, 53), (3, 53, 97), (20, 49, 97), (22, 49, 53), (49, 53, 97), (11, 49, 97), (46, 53, 97), (53, 67, 97)]
new neighbours at hop 2: [(9, 49, 53), (10, 53, 97), (20, 46, 97), (42, 49, 97), (53, 58, 97), (4, 49, 97), (49, 83, 97), (11, 46, 97), (53, 62, 97), (53, 73, 97), (41, 53, 97), (3, 49, 97), (49, 70, 97), (14, 46, 53), (44, 53, 97), (3, 20, 97), (14, 20, 49), (14, 53, 67), (3, 11, 97), (28, 53, 97), (4, 49, 53), (6, 49, 53), (53, 59, 97), (53, 74, 97), (10, 49, 97), (22, 46, 53), (9, 53, 97), (53, 72, 97), (20, 53, 97), (11, 22, 49), (3, 22, 53), (53, 76, 97), (35, 49, 97), (22, 53, 67), (17, 49, 97), (53, 89, 97), (49, 53, 72), (17, 53, 97), (7, 53, 97), (49, 92, 97), (49, 53, 90), (53, 93, 97), (49, 97, 99), (53, 86, 97), (4, 53, 97), (15, 53, 97), (5, 53, 97), (49, 53, 83), (18, 53, 97), (0, 53, 97), (11, 14, 49), (20, 67, 97), (27, 53, 97), (3, 14, 53), 

new neighbours at hop 5: [(33, 36, 53), (11, 49, 78), (28, 53, 89), (22, 48, 76), (42, 46, 61), (0, 12, 97), (11, 58, 83), (48, 49, 92), (33, 72, 97), (9, 38, 49), (3, 14, 23), (31, 53, 90), (11, 53, 89), (10, 14, 73), (3, 53, 99), (37, 93, 97), (28, 44, 97), (9, 49, 63), (14, 42, 76), (11, 46, 86), (56, 97, 99), (14, 35, 73), (14, 53, 90), (11, 83, 89), (2, 3, 22), (2, 22, 35), (3, 83, 99), (27, 67, 97), (9, 19, 49), (53, 66, 67), (46, 70, 72), (10, 14, 86), (1, 20, 22), (12, 26, 53), (14, 42, 89), (11, 27, 49), (3, 9, 42), (4, 22, 42), (11, 20, 46), (9, 48, 53), (6, 76, 97), (53, 54, 70), (3, 20, 56), (11, 22, 92), (33, 53, 73), (14, 28, 46), (20, 27, 72), (7, 89, 97), (3, 13, 53), (6, 27, 97), (6, 11, 89), (14, 20, 47), (6, 21, 53), (44, 53, 64), (30, 53, 89), (3, 11, 64), (14, 16, 49), (2, 12, 97), (42, 70, 97), (10, 22, 44), (3, 20, 69), (32, 53, 62), (22, 28, 42), (3, 4, 61), (10, 25, 49), (33, 53, 86), (0, 49, 72), (6, 51, 53), (18, 36, 53), (4, 61, 67), (8, 14, 20), (44, 53, 77

new neighbours at hop 6: [(0, 61, 97), (4, 67, 82), (42, 53, 64), (24, 67, 83), (17, 46, 95), (20, 33, 36), (43, 53, 68), (9, 49, 50), (11, 46, 73), (12, 22, 93), (0, 14, 22), (4, 28, 90), (6, 20, 85), (20, 95, 96), (7, 14, 72), (37, 63, 97), (20, 53, 99), (11, 39, 70), (29, 45, 53), (89, 94, 97), (42, 49, 66), (0, 53, 98), (8, 22, 44), (5, 18, 49), (39, 58, 97), (11, 61, 88), (0, 44, 49), (22, 41, 73), (20, 88, 93), (56, 67, 99), (28, 90, 92), (14, 47, 70), (7, 48, 70), (20, 46, 96), (7, 67, 83), (68, 73, 97), (4, 11, 42), (27, 86, 97), (11, 35, 72), (14, 24, 86), (4, 14, 47), (6, 28, 67), (10, 53, 65), (58, 67, 72), (11, 34, 76), (36, 80, 97), (22, 59, 63), (3, 35, 82), (19, 22, 35), (11, 25, 27), (11, 28, 32), (20, 83, 99), (14, 58, 84), (3, 6, 24), (18, 48, 61), (13, 34, 49), (13, 53, 62), (22, 30, 35), (4, 9, 90), (80, 81, 97), (6, 42, 86), (35, 74, 90), (23, 71, 97), (10, 67, 84), (3, 27, 83), (11, 49, 91), (13, 14, 70), (4, 41, 42), (11, 90, 92), (16, 72, 97), (4, 22, 29), (10, 

RuntimeError: false INTERNAL ASSERT FAILED at "/Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/BatchLinearAlgebra.cpp":1540, please report a bug to PyTorch. linalg.eigh: Argument 8 has illegal value. Most certainly there is a bug in the implementation calling the backend library.

In [ ]:
if Q <= 100:
    fig, ax = plt.subplots(1,2, figsize=(18, 5), gridspec_kw={'width_ratios': [1,1]})

    # -------------- Plot the Original Graph ----------------
    g = base_problem.underlying_graph
    nodes = g.nodes()
    colors = original_Y
    node_size = 150

    nx.draw_networkx(g, pos, with_labels=True, node_size=node_size, ax=ax[0])
    ec = nx.draw_networkx_edges(g, pos, alpha=0.2, ax=ax[0])
    nc = nx.draw_networkx_nodes(g, pos, nodelist=nodes, node_color=colors, ax=ax[0],
                                node_size=node_size*0.8, cmap=mpl.colormaps['Blues'])
    plt.colorbar(nc)
    ax[0].set_axis_off()

    # -------------- Plot a Random ComboSubgraph ----------------
    nodes = ComboSubgraph.nodes()
    colors = all_Y

    nx.draw_networkx(ComboSubgraph, pos_, with_labels=True, node_size=node_size, ax=ax[1])
    ec = nx.draw_networkx_edges(ComboSubgraph, pos_, alpha=0.2, ax=ax[1])
    nc = nx.draw_networkx_nodes(ComboSubgraph, pos_, nodelist=nodes, node_color=colors, ax=ax[1],
                                node_size=node_size*0.8, cmap=mpl.colormaps['Blues'])
    plt.colorbar(nc)

    ax[1].set_axis_off()

    plt.show()

### Compute results for all models

In [ ]:
kernels = ["polynomial","polynomial_suminverse","diffusion","diffusion_ard"]
models = []
models_r = []
sigma = 0.5
prop_train = 0.25

## Smooth signal
all_X, all_Y, X_train, Y_train, X_test, Y_test, X_best = get_signal(ComboSubgraph, original_Y, prop_data = prop_train)
for kernel in kernels:
    start_time = time.time()
    models.append(output_model(kernel, X_train, Y_train, ComboSubgraph)[0])
    end_time = time.time()
    print(f"{kernel} kernel finished in {round(end_time - start_time,1)}s")

## Rough signal
all_X_r, all_Y_r, X_train_r, Y_train_r, X_test_r, Y_test_r, X_best_r = get_signal_noise(ComboSubgraph, original_Y, 
                                                                                        prop_data = prop_train, sigma = sigma)
for kernel in kernels:
    start_time = time.time()
    models_r.append(output_model(kernel, X_train_r, Y_train_r, ComboSubgraph)[0])
    end_time = time.time()
    print(f"{kernel} kernel finished in {round(end_time - start_time,1)}s")

In [ ]:
fig = plt.figure(figsize=(20, 8), layout='constrained')
grid = plt.GridSpec(2, 3, hspace=0., wspace=0.2)

signal_ax = fig.add_subplot(grid[:, 0])
signal_ax.set_axis_off()
kernel_subplot = fig.add_subplot(grid[:, 1:])
kernel_subplot.set_axis_off()

inner_signal = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=signal_ax, wspace=0., hspace=0.2)
inner_kernel = gridspec.GridSpecFromSubplotSpec(4, 4, subplot_spec=kernel_subplot, wspace=0.2, hspace=0.5)

signal1_ax = plt.Subplot(fig, inner_signal[0])
signal2_ax = plt.Subplot(fig, inner_signal[1])

kernel11v_ax = plt.Subplot(fig, inner_kernel[0, 0])
kernel11v_ax.set_ylim(-3, 3)
kernel12v_ax = plt.Subplot(fig, inner_kernel[0, 1])
kernel12v_ax.set_ylim(-3, 3)
kernel13v_ax = plt.Subplot(fig, inner_kernel[0, 2])
kernel13v_ax.set_ylim(-3, 3)
kernel14v_ax = plt.Subplot(fig, inner_kernel[0, 3])
kernel14v_ax.set_ylim(-3, 3)

kernel11r_ax = plt.Subplot(fig, inner_kernel[1, 0])
kernel12r_ax = plt.Subplot(fig, inner_kernel[1, 1])
kernel13r_ax = plt.Subplot(fig, inner_kernel[1, 2])
kernel14r_ax = plt.Subplot(fig, inner_kernel[1, 3])

kernel21v_ax = plt.Subplot(fig, inner_kernel[2, 0])
kernel21v_ax.set_ylim(-3, 3)
kernel22v_ax = plt.Subplot(fig, inner_kernel[2, 1])
kernel22v_ax.set_ylim(-3, 3)
kernel23v_ax = plt.Subplot(fig, inner_kernel[2, 2])
kernel23v_ax.set_ylim(-3, 3)
kernel24v_ax = plt.Subplot(fig, inner_kernel[2, 3])
kernel24v_ax.set_ylim(-3, 3)

kernel21r_ax = plt.Subplot(fig, inner_kernel[3, 0])
kernel22r_ax = plt.Subplot(fig, inner_kernel[3, 1])
kernel23r_ax = plt.Subplot(fig, inner_kernel[3, 2])
kernel24r_ax = plt.Subplot(fig, inner_kernel[3, 3])


# Smooth Signal
plot_signal(signal1_ax, ComboSubgraph, all_Y, pos_, y_label="(a)")

plot_validation(kernel11v_ax, models[0], X_test, Y_test, title="Polynomial")
kernel11v_ax.set_ylabel("Validation prediction")
plot_rinverse(kernel11r_ax, models[0])
kernel11r_ax.set_ylabel(r'$r^{-1}(\lambda)$')
kernel11r_ax.set_xlabel(r'$\lambda$')
plot_validation(kernel12v_ax, models[1], X_test, Y_test, title="Sum of inverse polynomial")
plot_rinverse(kernel12r_ax, models[1])
kernel12r_ax.set_xlabel(r'$\lambda$')
plot_validation(kernel13v_ax, models[2], X_test, Y_test, title="Diffusion")
plot_rinverse(kernel13r_ax, models[2])
kernel13r_ax.set_xlabel(r'$\lambda$')
plot_validation(kernel14v_ax, models[3], X_test, Y_test, title="Diffusion with ARD")
plot_rinverse(kernel14r_ax, models[3])
kernel14r_ax.set_xlabel(r'$\lambda$')


## Rough Signal
plot_signal(signal2_ax, ComboSubgraph, all_Y_r, pos_, y_label="(b)")

plot_validation(kernel21v_ax, models_r[0], X_test_r, Y_test_r)
kernel21v_ax.set_ylabel("Validation prediction")
plot_rinverse(kernel21r_ax, models_r[0])
kernel21r_ax.set_ylabel(r'$r^{-1}(\lambda)$')
kernel21r_ax.set_xlabel(r'$\lambda$')
plot_validation(kernel22v_ax, models_r[1], X_test_r, Y_test_r)
plot_rinverse(kernel22r_ax, models_r[1])
kernel22r_ax.set_xlabel(r'$\lambda$')
plot_validation(kernel23v_ax, models_r[2], X_test_r, Y_test_r)
plot_rinverse(kernel23r_ax, models_r[2])
kernel23r_ax.set_xlabel(r'$\lambda$')
plot_validation(kernel24v_ax, models_r[3], X_test_r, Y_test_r)
plot_rinverse(kernel24r_ax, models_r[3])
kernel24r_ax.set_xlabel(r'$\lambda$')

fig.add_subplot(signal1_ax)
fig.add_subplot(signal2_ax)

fig.add_subplot(kernel11v_ax)
fig.add_subplot(kernel12v_ax)
fig.add_subplot(kernel13v_ax)
fig.add_subplot(kernel14v_ax)
fig.add_subplot(kernel11r_ax)
fig.add_subplot(kernel12r_ax)
fig.add_subplot(kernel13r_ax)
fig.add_subplot(kernel14r_ax)


fig.add_subplot(kernel21v_ax)
fig.add_subplot(kernel22v_ax)
fig.add_subplot(kernel23v_ax)
fig.add_subplot(kernel24v_ax)
fig.add_subplot(kernel21r_ax)
fig.add_subplot(kernel22r_ax)
fig.add_subplot(kernel23r_ax)
fig.add_subplot(kernel24r_ax)

save_dir = './Figures/KernelValidation'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

plt.savefig(f"{save_dir}/KernelValidation-{config['problem_settings']['graph_type']}"
            f"-n{config['problem_settings']['n']}-k{k}-Q{Q}-j{j}-sigma{sigma}-prop{prop_train}.pdf", bbox_inches='tight')
plt.show()